# DVC: Data Version Control

## What is DVC?
DVC (Data Version Control) is an open-source tool that brings Git-like versioning to data, models, and ML pipelines. It works alongside Git, storing large files in remote storage while tracking them via lightweight pointer files in Git.

## Why DVC?
- Git handles code, DVC handles data & models
- Large files (GBs) cannot go into Git
- Reproducible ML pipelines
- Collaboration on datasets and experiments

## Core Concepts
| Concept | Description |
|---------|------------|
| `.dvc` files | Lightweight pointer files tracked in Git |
| `dvc.yaml` | Pipeline definition (stages, deps, outs) |
| `dvc.lock` | Pipeline execution state (like lockfile) |
| Remote | Storage backend (S3, GCS, Azure, local) |
| Cache | Local cache of DVC-tracked files |

In [1]:
# Installation
# pip install dvc dvc-s3 dvc-gs dvc-azure dvc-gdrive

# Verify installation
import subprocess
result = subprocess.run(['dvc', '--version'], capture_output=True, text=True)
print(f"DVC version: {result.stdout.strip()}")

DVC version: 3.67.1


## DVC Initialization & Basic Commands

```bash
# Initialize DVC in a Git repo
git init my_ml_project
cd my_ml_project
dvc init
git commit -m 'Initialize DVC'

# Track a data file
dvc add data/dataset.csv
git add data/dataset.csv.dvc data/.gitignore
git commit -m 'Add dataset'

# Track a directory
dvc add data/
git add data.dvc
git commit -m 'Track data directory'
```

In [2]:
# Simulate DVC workflow
import os, json, hashlib, pandas as pd, numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import joblib

os.makedirs('/tmp/dvc_demo/data', exist_ok=True)
os.makedirs('/tmp/dvc_demo/models', exist_ok=True)

# Create dataset
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df.to_csv('/tmp/dvc_demo/data/iris.csv', index=False)
print(f'Dataset saved: {df.shape}')
print(df.head())

# Simulate a .dvc pointer file (what DVC actually stores in Git)
file_content = open('/tmp/dvc_demo/data/iris.csv', 'rb').read()
md5_hash = hashlib.md5(file_content).hexdigest()
dvc_pointer = {
    'outs': [{
        'md5': md5_hash,
        'size': len(file_content),
        'path': 'data/iris.csv'
    }]
}
print('\nDVC pointer file (.dvc):')   
print(json.dumps(dvc_pointer, indent=2))

Dataset saved: (150, 5)
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  

DVC pointer file (.dvc):
{
  "outs": [
    {
      "md5": "4d301abed5efe50eccda350cde38e0eb",
      "size": 2777,
      "path": "data/iris.csv"
    }
  ]
}


## Remote Storage Setup

```bash
# Local remote (good for testing)
dvc remote add -d myremote /tmp/dvcstore

# Amazon S3
dvc remote add -d s3remote s3://mybucket/dvcstore
dvc remote modify s3remote region us-east-1

# Google Cloud Storage
dvc remote add -d gsremote gs://mybucket/dvcstore

# Azure Blob Storage
dvc remote add -d azremote azure://mycontainer/dvcstore
dvc remote modify azremote connection_string 'DefaultEndpointsProtocol=...' 

# SSH remote
dvc remote add -d sshremote ssh://user@server/path/to/store

# Push/pull data
dvc push          # upload tracked files to remote
dvc pull          # download tracked files from remote
dvc fetch         # download to cache without checkout
```

## DVC Pipelines

DVC pipelines define reproducible ML workflows as a DAG (Directed Acyclic Graph).

### dvc.yaml Structure
```yaml
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
      - src/prepare.py
      - data/raw/iris.csv
    outs:
      - data/processed/train.csv
      - data/processed/test.csv
    params:
      - params.yaml:
          - split.test_size
          - split.random_state

  train:
    cmd: python src/train.py
    deps:
      - src/train.py
      - data/processed/train.csv
    outs:
      - models/model.pkl
    params:
      - params.yaml:
          - train.n_estimators
          - train.max_depth

  evaluate:
    cmd: python src/evaluate.py
    deps:
      - src/evaluate.py
      - models/model.pkl
      - data/processed/test.csv
    metrics:
      - metrics/scores.json:
          cache: false
    plots:
      - metrics/confusion_matrix.csv:
          cache: false
```

```bash
# Run pipeline (only runs changed stages)
dvc repro

# Force re-run all stages
dvc repro --force

# Visualize DAG
dvc dag
```

In [3]:
# Simulate pipeline stages
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# --- Stage 1: prepare ---
df = pd.read_csv('/tmp/dvc_demo/data/iris.csv')
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df.to_csv('/tmp/dvc_demo/data/train.csv', index=False)
test_df.to_csv('/tmp/dvc_demo/data/test.csv', index=False)
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

# --- Stage 2: train ---
X_train = train_df[iris.feature_names].values
y_train = train_df['target'].values

params = {'n_estimators': 100, 'max_depth': 5, 'random_state': 42}
model = RandomForestClassifier(**params)
model.fit(X_train, y_train)
joblib.dump(model, '/tmp/dvc_demo/models/model.pkl')
print(f'Model saved')

# --- Stage 3: evaluate ---
X_test = test_df[iris.feature_names].values
y_test = test_df['target'].values

model = joblib.load('/tmp/dvc_demo/models/model.pkl')
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

scores = {'accuracy': acc, 'test_samples': len(y_test)}
with open('/tmp/dvc_demo/scores.json', 'w') as f:
    json.dump(scores, f, indent=2)
print(f'Accuracy: {acc:.4f}')
print('Scores:', json.dumps(scores, indent=2))

Train: 120 | Test: 30


Model saved
Accuracy: 1.0000
Scores: {
  "accuracy": 1.0,
  "test_samples": 30
}


## DVC Experiments

```bash
# Run an experiment
dvc exp run

# Run with modified params
dvc exp run --set-param train.n_estimators=200

# Show all experiments
dvc exp show

# Compare experiments
dvc exp diff

# Apply best experiment
dvc exp apply <exp_name>

# Save experiment as a branch
dvc exp branch <exp_name> my-feature-branch

# Garbage collect old experiments
dvc exp gc --workspace
```

## DVC with Git Workflow

```bash
# Full workflow example
git init my_project && cd my_project
dvc init

# Add data
dvc add data/train.csv
git add data/train.csv.dvc data/.gitignore
git commit -m 'feat: add training data v1'

# Update data (new version)
# ... modify data/train.csv ...
dvc add data/train.csv
git add data/train.csv.dvc
git commit -m 'feat: update training data to v2'

# Switch back to data v1
git checkout HEAD~1 data/train.csv.dvc
dvc checkout

# Collaboration
dvc push                    # share data
git push                    # share code + .dvc pointers

# Teammate
git pull
dvc pull                    # get data
```

## Additional Learning Resources

### Official Documentation
- [DVC Docs](https://dvc.org/doc) Complete guide
- [DVC Get Started](https://dvc.org/doc/start)
- [DVC Pipelines](https://dvc.org/doc/user-guide/project-structure/dvcyaml-files)
- [DVC Experiments](https://dvc.org/doc/user-guide/experiment-management)

### Tutorials
- [MLOps Zoomcamp DVC module](https://github.com/DataTalksClub/mlops-zoomcamp)
- [Iterative.ai Blog](https://iterative.ai/blog)
- [DVC YouTube channel](https://www.youtube.com/@DVCorg)

### Papers & Articles
- [DVC: Version Control for Machine Learning](https://arxiv.org/abs/2209.08088)